In [ ]:
from datos_simulacion import *

In [7]:
class Celda:
    def __init__(self, estado, tiempo_quema, tiempo_humedad):
        self._estado = estado
        self._tiempo_quema = tiempo_quema
        self._tiempo_humedad = tiempo_humedad
    
    def decrease_quema_one_time_unit(self):
        self._tiempo_quema -= 1
    def decrease_humedad_one_time_unit(self):
        self._tiempo_humedad -= 1
    def change_state(self, new_state):
        self._estado = new_state
    

In [8]:
import numpy as np

class CellularFireAutomata:
    def __init__(self, grid_estados, grid_quema, grid_humedad):
        # los datos de la entrada estan en formato np.array
        self._grid_estados = grid_estados
        self._grid_quema = grid_quema
        self._grid_humedad = grid_humedad
        self._m, self._n = grid_estados.shape
        self._global_grid = np.empty((self._m, self._n), dtype=object)
        for i in range(self._m):
            for j in range(self._n):
                c = Celda(grid_estados[i][j], grid_quema[i][j], grid_humedad[i][j])
                self._global_grid[i, j] = c
    
    def nucli(self, i, j):
        return self._global_grid[i, j]
    
    def vecindad_moore(self, i, j):
        # en los bordes se consideran celdas sanas que nunca pueden cambiar su estado
        vecinos = []
        for di in [-1, 0, 1]:
            for dj in [-1, 0, 1]:
                # Saltar la celda central
                if di == 0 and dj == 0:
                    continue
                
                # Comprobar límites del grid
                ni, nj = i + di, j + dj
                if 0 <= ni < self._m and 0 <= nj < self._n:
                    vecinos.append(self._global_grid[ni, nj])
                else:
                    # Crear una celda "sana" para los bordes (estado 0, sin cambios)
                    vecinos.append(Celda(0, 9999, 9999))  # Valores grandes para que nunca cambien
        
        return vecinos
    
    def actualizar(self):
        # Crear un nuevo grid para almacenar los estados actualizados
        nuevo_grid = np.empty((self._m, self._n), dtype=object)
        
        # Actualizar cada celda basándose en sus vecinos
        for i in range(self._m):
            for j in range(self._n):
                celda_actual = self.nucli(i, j)
                vecinos = self.vecindad_moore(i, j)
                
                # Crear nueva celda con los mismos valores iniciales
                nueva_celda = Celda(celda_actual._estado, 
                                    celda_actual._tiempo_quema, 
                                    celda_actual._tiempo_humedad)
                
                # Aplicar reglas de actualización
                
                # Regla 1: Si ya está quemada (estado 2), se mantiene así
                if celda_actual._estado == 2:  # QUEMADO
                    pass  # No hay cambios
                
                # Regla 2: Si se está quemando (estado 1), reducir tiempo de quema
                elif celda_actual._estado == 1:  # QUEMANDOSE
                    nueva_celda.decrease_quema_one_time_unit()
                    
                    # Si se acabó el tiempo de quema, pasar a QUEMADO (estado 2)
                    if nueva_celda._tiempo_quema <= 0:
                        nueva_celda.change_state(2)  # QUEMADO
                
                # Regla 3: Si no está quemada (estado 0), comprobar vecinos y humedad
                elif celda_actual._estado == 0:  # NO_QUEMADO
                    # Comprobar si hay algún vecino quemándose
                    tiene_vecino_quemandose = any(vecino._estado == 1 for vecino in vecinos)
                    
                    # Si hay un vecino quemándose, reducir humedad
                    if tiene_vecino_quemandose:
                        nueva_celda.decrease_humedad_one_time_unit()
                        
                        # Si la humedad llega a cero, la celda empieza a quemarse
                        if nueva_celda._tiempo_humedad <= 0:
                            nueva_celda.change_state(1)  # QUEMANDOSE
                
                # Guardar la celda actualizada en el nuevo grid
                nuevo_grid[i, j] = nueva_celda
        
        # Actualizar el grid global con los nuevos estados
        self._global_grid = nuevo_grid

    def get_estado_matrix(self):
        return self._grid_estados
    
    def get_humedad_matrix(self):
        return self._grid_humedad
    
    def get_quema_matrix(self):   
        return self._grid_quema

In [9]:
import pygame
import time
import sys
# Clase para la visualización con Pygame
class FireSimulationVisualizer:
    # Colores
    COLOR_BACKGROUND = (50, 50, 50)
    COLOR_NO_QUEMADO = (34, 139, 34)  # Verde bosque
    COLOR_QUEMANDOSE = (255, 69, 0)   # Rojo-naranja
    COLOR_QUEMADO = (128, 128, 128)   # Gris
    
    def __init__(self, automata, cell_size=10, fps=1):
        self.automata = automata
        self.cell_size = cell_size
        self.fps = fps
        self.width = automata._n * cell_size
        self.height = automata._m * cell_size
        
        # Inicializar Pygame
        pygame.init()
        self.screen = pygame.display.set_mode((self.width, self.height))
        pygame.display.set_caption("Simulación de Propagación de Fuego")
        self.clock = pygame.time.Clock()
        self.font = pygame.font.SysFont(None, 24)
        
    def get_cell_color(self, estado, tiempo_humedad, tiempo_quema):
        if estado == 2:  # QUEMADO
            return self.COLOR_QUEMADO
        elif estado == 1:  # QUEMANDOSE
            # Variación de color según tiempo de quema (más intenso al principio)
            intensidad = min(255, 150 + tiempo_quema * 10)
            return (intensidad, 69, 0)
        else:  # NO_QUEMADO
            # Variación de verde según humedad (más oscuro = más húmedo)
            intensidad = max(34, 120 - tiempo_humedad // 2)
            return (34, intensidad, 34)
    
    def draw_grid(self):

        for i in range(self.automata._m):
            for j in range(self.automata._n):
                celda = self.automata.nucli(i, j)
                color = self.get_cell_color(celda._estado, celda._tiempo_humedad, celda._tiempo_quema)
                
                rect = pygame.Rect(j * self.cell_size, self.height - (i + 1) * self.cell_size, 

                                self.cell_size, self.cell_size)
                pygame.draw.rect(self.screen, color, rect)
                pygame.draw.rect(self.screen, (20, 20, 20), rect, 1)
    
    def draw_info(self, generation, elapsed_time):
        info_text = f"Generación: {generation} | Tiempo: {elapsed_time:.1f}s"
        text_surface = self.font.render(info_text, True, (255, 255, 255))
        self.screen.blit(text_surface, (10, 10))
    
    def run(self, max_generations=500):
        generation = 0
        start_time = time.time()
        running = True
        
        while running and generation < max_generations:
            # Manejo de eventos
            for event in pygame.event.get():
                if event.type == pygame.QUIT:
                    running = False
                elif event.type == pygame.KEYDOWN:
                    if event.key == pygame.K_ESCAPE:
                        running = False
                    elif event.key == pygame.K_SPACE:
                        # Pausar/Reanudar
                        paused = True
                        while paused:
                            for evt in pygame.event.get():
                                if evt.type == pygame.KEYDOWN and evt.key == pygame.K_SPACE:
                                    paused = False
                                elif evt.type == pygame.QUIT:
                                    paused = False
                                    running = False
                            time.sleep(0.1)
            
            # Limpiar pantalla
            self.screen.fill(self.COLOR_BACKGROUND)
            
            # Dibujar el estado actual
            self.draw_grid()
            
            # Mostrar información
            elapsed_time = time.time() - start_time
            self.draw_info(generation, elapsed_time)
            
            # Actualizar pantalla
            pygame.display.flip()
            
            # Actualizar el autómata celular
            self.automata.actualizar()
            generation += 1
            
            # Control de velocidad
            self.clock.tick(self.fps)
            
            # Verificar si el fuego se ha extinguido
            estados = self.automata.get_estado_matrix()
            if not 1 in estados:  # No hay celdas quemándose
                if generation > 10:  # Evitar terminar demasiado pronto
                    break
        
        # Mensaje final
        if generation >= max_generations:
            print(f"Simulación terminada después de {max_generations} generaciones")
        else:
            print(f"Fuego extinguido después de {generation} generaciones")
        
        # Mantener la ventana abierta hasta que el usuario la cierre
        waiting = True
        while waiting and running:
            for event in pygame.event.get():
                if event.type == pygame.QUIT or (event.type == pygame.KEYDOWN and event.key == pygame.K_ESCAPE):
                    waiting = False
            time.sleep(0.1)
        
        pygame.quit()

In [10]:
automata = CellularFireAutomata(estado_vegetacion, tiempo_quema, proteccion_humedad)
    
# Crear y ejecutar el visualizador
visualizer = FireSimulationVisualizer(automata, cell_size=12, fps=3)
visualizer.run()

Fuego extinguido después de 181 generaciones
